# **Comprehensive Fine-Mapping Framework**

Hi everyone. Today, I’m going to share with you our updated fine-mapping framework, which performs an automated QC on results generated by our existing pipeline and then diverts to better models based on a diagnostics table.  
The goal of this improvement is to **diminish the effect of LD mismatch**.

---

## **Background**

Before walking you through the updated framework, it's important to understand the current SuSiE_RSS pipeline and why it is sensitive to LD mismatch.

### 🔍 **SuSiE_RSS**

**SuSiE_RSS** extends the Sum of Single Effects model (SuSiE) to **summary statistics**. More details are displayed in this [paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC9337707/)

Although most people here are familiar with SuSiE, I’ll briefly reintroduce it to keep the logic coherent.

In short, SuSiE models perform linear regression with **smart variable selection**, focusing on quantifying uncertainty.  
It has two key parameters:
- **L**: number of single effects
- **max_iter**: maximum number of iterations

SuSiE aims to build a model with L credible sets (CSs).  
Each iteration:
- Fits a **Single Effect Regression (SER)** model
- Updates residuals by removing the effect of the newly found CS
- Repeats until convergence

**Challenges:**
- SuSiE_RSS uses the LD matrix **repeatedly** at every iteration.
- If the LD matrix is inaccurate, **errors compound across iterations**, pulling the model away from true signals.

Depending on the settings of L and max_iter, we get three models:

| Model | L | max_iter | Description |
|:---|:---|:---|:---|
| **BVSR** (Bayesian Variable Selection Regression) | L | adaptive | Full iterative, conditions on other effects |
| **BCR** (Bayesian Conditional Regression) | L | 1 | No convergence; robust to LD mismatch |
| **SER** (Single Effect Regression) | 1 | 1 | Assumes only one causal variant |

---

### **Current Pipeline**

| Step | Description |
|:---|:---|
| 1 | Imputation (using LD reference once) |
| 2 | Basic QC (allele flipping check) |
| 3 | SuSiE_RSS QC (`rss_qc`) (uses LD reference multiple times) |
| 4 | BVSR (fit using cleaned summary statistics) |

`rss_qc` does two important things:
- **Outlier Detection**: flags SNPs where z-scores mismatch the LD
- **LD Discrepancy Correction**: removes variants inflating or deflating signals

This preprocessing step is crucial for improving SuSiE_RSS accuracy and stability.

---

## **🧠 Other Models in the Pipeline**

In addition to SuSiE_RSS, we use:
- **BVSR**: flexible, many iterations, captures multiple signals
- **BCR**: simpler, 1 iteration, robust to imperfect LD
- **SER**: simplest, assumes a single causal variant

---

# **📐 Updated Framework: Overview**

Here’s the flow of the updated fine-mapping framework:

---

## **1. Run BVSR**

- Use summary statistics that have been imputed, basic-QCed, and SuSiE_RSS-QCed.

---

## **2. Extract Key Information from BVSR**

- Identify study blocks that contain:
  - Credible Sets (CSs), or
  - Variants with **PIP > 0.025**.

---

## **3. Create a Diagnostic Table**

Summarize important metrics, including:
- Top variant p-value
- Correlation between credible sets (cs_corr)
- Number of variants per CS
- Other quality metrics

---

## **4. Logic for Adding Additional Analyses**

| Condition | Action |
|:---|:---|
| CS with p-value > 1e-4, OR cs_corr > 0.5 with another CS | Add **SER** and **BCR** |
| Only one CS, OR no CS but variants with PIP > 0.025 | Add **SER** only |

---

## **5. Logic for Exporting specific model**
### Automated QC Definitions
| Term | Definition |
|:---|:---|
| **top_cs** | The CS with the largest absolute z-score |
| **flagged_cs** | CS meeting any of:<br>• Highly correlated with top_cs<br>• Top variant p-value high |
| **remaining_cs** | The remaining CSs after excluding top_cs and flagged_cs |

### Retention Logic

| Situation | Model Retention |
|:---|:---|
| Only one CS in study block | Keep **BVSR** only |
| No flagged CSs but have remaining CS | Keep **BVSR** only |
| No remaining CSs after excluding top_cs and flagged_cs | Keep **BVSR** and **SER** |
| Remaining CSs still exist | Keep **BVSR**, **SER**, and **BCR** |

- Export results from the best models based on diagnostics.

---

# **❓ Is Imputation Robust to LD Inaccuracies?**

**Short answer**: Somewhat robust — but not fully immune.

| Feature | Benefit |
|:---|:---|
| **Regularization (λ)** | Adds a small constant to LD diagonal, stabilizing inversion (ridge regression-like) |
| **Filtering (rcond)** | Discards tiny eigenvalues to avoid overfitting |
| **R² thresholding** | Removes low-quality SNPs |

All these tricks help stabilize modeling even when the LD matrix is imperfect.

---

# **Thank you! 🙌**

---
